# MR Similarity Analysis - Unified Formula

---

## Similarity Computation Summary

### Unified Formula

The similarity between two MRs is computed using the following formula, **shared across all domains**:

```
Composite = 0.100 × J_LHS +           # 10% - Jaccard on LHS elements (no direction)
            0.100 × J_RHS +           # 10% - Jaccard on RHS elements (unsigned)
            0.375 × J_LHS_Signed +    # 37.5% - Jaccard on (element, direction) in the LHS
            0.375 × J_RHS_Signed +    # 37.5% - Jaccard on (element, sign) in the RHS
            0.050 × Sim_Template      # 5% - identical structure (1 or 0)
```

### Jaccard Index

For each pair of sets A and B:

```
Jaccard(A, B) = |A ∩ B| / |A ∪ B|
```

* If both sets are empty: Jaccard = 1.0
* If there is no intersection: Jaccard = 0.0
* If they are identical: Jaccard = 1.0

---

### Domain-Specific Element Extraction

#### NUMERIC Domain (AVs, DCs)

| Component      | Description       | Example                           |
| -------------- | ----------------- | --------------------------------- |
| **LHS**        | Unsigned features | `{NS}`, `{CPU, W}`                |
| **RHS**        | Unsigned features | `{TTD, D}`, `{T}`                 |
| **LHS_Signed** | (feature, sign)   | `{(NS, -)}`, `{(CPU, -), (W, 0)}` |
| **RHS_Signed** | (feature, sign)   | `{(TTD, +), (D, +)}`              |

Where the sign is derived from the operator:

* `>=`, `>` → `+` (positive)
* `<=`, `<` → `-` (negative)
* `==`, `!=` → `0` (neutral)

**Canonicalization:** `NS(m1) > NS(m2)` is transformed into `NS(m2) < NS(m1)` so both forms are treated as equivalent.

**NUMERIC Example:**

```
MR: NS(m1) > NS(m2) => TTD(m2) >= TTD(m1) AND D(m2) >= D(m1)

After canonicalizing to m2 op m1:
  LHS: NS(m2) < NS(m1)  →  feature=NS, sign=-
  RHS: TTD(m2) >= TTD(m1), D(m2) >= D(m1)  →  (TTD,+), (D,+)

Signature:
  LHS = {NS}
  RHS = {TTD, D}
  LHS_Signed = {(NS, -)}
  RHS_Signed = {(TTD, +), (D, +)}
  Template = "LHS[1]=>RHS[2]"
```

---

#### PREDICATE Domain (DFAs, Words)

| Component      | Description                     | Example                                |
| -------------- | ------------------------------- | -------------------------------------- |
| **LHS**        | Normalized operations           | `{FS()->size(), NFS()}`                |
| **RHS**        | Normalized predicates           | `{R(), ¬R()}`                          |
| **LHS_Signed** | (operation, direction|operator) | `{(FS()->includesAll(), x2->x1\|>)}`   |
| **RHS_Signed** | (predicate, full signature)     | `{(R(), R(x1,w1)), (¬R(), ¬R(x2,w1))}` |

**Directionality is preserved:**

* `FS(x2)->includesAll(FS(x1))` → `(FS()->includesAll(), x2->x1|>)`
* `FS(x1)->includesAll(FS(x2))` → `(FS()->includesAll(), x1->x2|>)`

These TWO expressions are **different** because the subset direction is opposite.

**PREDICATE Example (DFAs):**

```
MR: FS(x2)->includesAll(FS(x1)) => ¬R(x1, w1) or R(x2, w1)

Signature:
  LHS = {FS()->includesAll(), FS()}
  RHS = {¬R(), R()}
  LHS_Signed = {(FS()->includesAll(), x2->x1|>), (FS(), x1|>)}
  RHS_Signed = {(¬R(), ¬R(x1,w1)), (R(), R(x2,w1))}
  Template = "LHS[2]=>RHS[2]"
```

**PREDICATE Example (Words):**

```
MR: w1->last() == '0' and w1->including('1') == w2 => R(x1, w1) and ¬R(x1, w2)

Signature:
  LHS = {w->last(), w->including()}
  RHS = {R(), ¬R()}
  LHS_Signed = {(w->last(), w1|==), (w->including(), w1->'1'|==)}
  RHS_Signed = {(R(), R(x1,w1)), (¬R(), ¬R(x1,w2))}
  Template = "LHS[2]=>RHS[2]"
```

---

### Full Computation Example

**MR1:** `NS(m1) > NS(m2) => TTD(m2) >= TTD(m1)`
**MR2:** `OC(m1) > OC(m2) => TTD(m2) >= TTD(m1)`

```
Signatures:
  MR1: LHS={NS}, RHS={TTD}, LHS_Signed={(NS,-)}, RHS_Signed={(TTD,+)}
  MR2: LHS={OC}, RHS={TTD}, LHS_Signed={(OC,-)}, RHS_Signed={(TTD,+)}

Jaccard computations:
  J_LHS        = |{NS} ∩ {OC}| / |{NS} ∪ {OC}| = 0/2 = 0.0
  J_RHS        = |{TTD} ∩ {TTD}| / |{TTD}| = 1/1 = 1.0
  J_LHS_Signed = |{(NS,-)} ∩ {(OC,-)}| / |...| = 0/2 = 0.0
  J_RHS_Signed = |{(TTD,+)} ∩ {(TTD,+)}| / |...| = 1/1 = 1.0
  Sim_Template = 1.0 (both "LHS[1]=>RHS[1]")

Composite = 0.10×0 + 0.10×1 + 0.375×0 + 0.375×1 + 0.05×1
          = 0 + 0.1 + 0 + 0.375 + 0.05
          = 0.525
```

---

### Redundancy Detection

* **Pairs >= 0.99**: considered **almost identical**
* **Pairs >= 0.60**: considered **potentially redundant**
* Redundant MRs are removed while keeping the first occurrence


## Imports


In [341]:
#!/usr/bin/env python3
"""
MR Similarity Analysis Tool - Unified Formula Version
======================================================
Fórmula de similaridad común para todos los dominios:

Composite = 0.10 * J_LHS +           # Jaccard operaciones LHS (sin signo)
            0.10 * J_RHS +           # Jaccard operaciones RHS (sin signo)
            0.375 * J_LHS_Signed +   # Jaccard (operación, dirección) LHS
            0.375 * J_RHS_Signed +   # Jaccard (operación, signo) RHS
            0.05 * Sim_Template      # Estructura igual

DOMINIOS:
- NUMERIC: AVs, DCs - comparaciones Feature(m1) op Feature(m2)
- PREDICATE: DFAs, Words - operaciones OCL + predicados R(x,w)
"""

import numpy as np
import pandas as pd
from dataclasses import dataclass, field
from typing import List, Set, Tuple, Dict, Optional
from enum import Enum
from collections import Counter
import matplotlib.pyplot as plt
import seaborn as sns
import re
import os
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')


## Configuration


In [343]:
INPUT_FILES = [
    "mrs_AVs.txt",
    "mrs_DCs_redundant.txt",
    "mrs_DFAs_redundant.txt",
    "mrs_Words.txt",
]

OUTPUT_DIR = "output_similarity"

HEATMAP_FIGSIZE = (14, 12)
HEATMAP_ANNOT = True
HEATMAP_FMT = '.2f'
REDUNDANCY_THRESHOLD = 0.6

# Pesos de la fórmula unificada (basada en la original)
WEIGHTS = {
    'J_LHS': 0.10,
    'J_RHS': 0.10,
    'J_LHS_Signed': 0.375,
    'J_RHS_Signed': 0.375,
    'Sim_Template': 0.05
}

os.makedirs(OUTPUT_DIR, exist_ok=True)


## Data Model


In [345]:
class ComparisonOperator(Enum):
    GT = ">"
    GE = ">="
    LT = "<"
    LE = "<="
    EQ = "=="
    NE = "!="

class Sign(Enum):
    POSITIVE = "+"
    NEGATIVE = "-"
    NEUTRAL = "0"

class Domain(Enum):
    NUMERIC = "numeric"      # AVs, DCs
    PREDICATE = "predicate"  # DFAs, Words

@dataclass
class MetamorphicRelation:
    """MR completa con información de dominio"""
    id: str
    group: str = "default"
    category: str = "normal"
    domain: Domain = Domain.NUMERIC
    lhs_raw: str = ""
    rhs_raw: str = ""
    raw_text: str = ""
    
    # Elementos extraídos (comunes a todos los dominios)
    lhs_elements: List[str] = field(default_factory=list)        # Operaciones/features sin signo
    rhs_elements: List[str] = field(default_factory=list)        # Operaciones/features sin signo
    lhs_signed: List[Tuple[str, str]] = field(default_factory=list)  # (operación, dirección)
    rhs_signed: List[Tuple[str, str]] = field(default_factory=list)  # (operación, signo)
    
    @property
    def full_id(self) -> str:
        return f"{self.group}-{self.category}-{self.id}"

@dataclass
class UnifiedSignature:
    """Signature unificada para todos los dominios"""
    mr_id: str
    lhs: Set[str] = field(default_factory=set)                    # Elementos LHS sin signo
    rhs: Set[str] = field(default_factory=set)                    # Elementos RHS sin signo
    lhs_signed: Set[Tuple[str, str]] = field(default_factory=set) # (elemento, dirección)
    rhs_signed: Set[Tuple[str, str]] = field(default_factory=set) # (elemento, signo)
    template: str = ""


## Domain Detector


In [347]:
def detect_domain(text: str, filename: str = "") -> Domain:
    """Detecta el dominio basándose en el contenido y nombre del archivo"""
    filename_lower = filename.lower()
    
    # Por nombre de archivo
    if 'av' in filename_lower or 'dc' in filename_lower:
        return Domain.NUMERIC
    if 'dfa' in filename_lower or 'word' in filename_lower:
        return Domain.PREDICATE
    
    # Por contenido
    has_m_vars = bool(re.search(r'\bm[12]\b', text))
    has_predicates = bool(re.search(r'[¬!]?\s*R(esult)?\s*\(', text))
    
    if has_predicates:
        return Domain.PREDICATE
    elif has_m_vars:
        return Domain.NUMERIC
    
    return Domain.NUMERIC  # Default


## Parser: Numeric Domain


In [349]:
class NumericParser:
    """Parser para MRs del dominio numérico (AVs, DCs)"""
    
    OPERATOR_MAP = {
        '>=': ComparisonOperator.GE, '<=': ComparisonOperator.LE,
        '>': ComparisonOperator.GT, '<': ComparisonOperator.LT,
        '==': ComparisonOperator.EQ, '!=': ComparisonOperator.NE,
        '=': ComparisonOperator.EQ,
    }
    
    def get_sign(self, operator: ComparisonOperator) -> str:
        """Obtiene el signo (+/-/0) de un operador"""
        if operator in [ComparisonOperator.GE, ComparisonOperator.GT]:
            return "+"
        elif operator in [ComparisonOperator.LE, ComparisonOperator.LT]:
            return "-"
        return "0"
    
    def parse_comparison(self, text: str) -> Optional[Tuple[str, ComparisonOperator]]:
        """Parsea una comparación como TTD(m2) >= TTD(m1)"""
        text = text.strip()
        
        match = re.search(r'(>=|<=|>(?!=)|<(?!=)|==|!=|=)', text)
        if not match:
            return None
        
        op_str = match.group(1)
        operator = self.OPERATOR_MAP[op_str]
        left, right = text.split(op_str, 1)
        left, right = left.strip(), right.strip()
        
        # Patrón Feature(m1) o Feature(m2)
        pattern = r'(\w+)\s*\(\s*(m[12])\s*\)'
        left_m = re.match(pattern, left)
        right_m = re.match(pattern, right)
        
        if left_m and right_m:
            left_feat, left_model = left_m.groups()
            right_feat, right_model = right_m.groups()
            
            if left_feat != right_feat:
                return None
            
            # Canonicalizar a m2 op m1
            is_m2_left = (left_model == 'm2')
            if not is_m2_left:
                flip = {
                    ComparisonOperator.GT: ComparisonOperator.LT,
                    ComparisonOperator.LT: ComparisonOperator.GT,
                    ComparisonOperator.GE: ComparisonOperator.LE,
                    ComparisonOperator.LE: ComparisonOperator.GE
                }
                operator = flip.get(operator, operator)
            
            return (left_feat, operator)
        
        return None
    
    def parse_mr(self, lhs: str, rhs: str, group: str, category: str, mr_id: str) -> MetamorphicRelation:
        """Parsea una MR del dominio numérico"""
        mr = MetamorphicRelation(
            id=mr_id, group=group, category=category,
            domain=Domain.NUMERIC,
            lhs_raw=lhs, rhs_raw=rhs,
            raw_text=f"{lhs} => {rhs}"
        )
        
        # Parsear LHS
        for part in re.split(r'\s+AND\s+', lhs, flags=re.I):
            result = self.parse_comparison(part.strip())
            if result:
                feature, operator = result
                mr.lhs_elements.append(feature)
                mr.lhs_signed.append((feature, self.get_sign(operator)))
        
        # Parsear RHS
        for part in re.split(r'\s+AND\s+', rhs, flags=re.I):
            result = self.parse_comparison(part.strip())
            if result:
                feature, operator = result
                mr.rhs_elements.append(feature)
                mr.rhs_signed.append((feature, self.get_sign(operator)))
        
        return mr

# ============================================================
# PARSER: PREDICATE DOMAIN (DFAs + Words unificados)
# ============================================================

class PredicateParser:
    """Parser para MRs del dominio de predicados (DFAs, Words)"""
    
    def extract_lhs_operations(self, lhs: str) -> Tuple[List[str], List[Tuple[str, str]]]:
        """
        Extrae operaciones del LHS preservando direccionalidad.
        
        Retorna:
            - elements: lista de operaciones sin dirección (para J_LHS)
            - signed: lista de (operación, dirección) (para J_LHS_Signed)
        """
        elements = []
        signed = []
        
        # Patrón para operaciones con método: FS(x1)->size(), w1->last()
        # Grupo 1: base (FS, NFS, T, w1, w2, etc.)
        # Grupo 2: variable si es función (x1, x2)
        # Grupo 3: método (size, last, includesAll, etc.)
        # Grupo 4: argumento del método si existe
        
        # Primero buscar operaciones tipo FS(x)->method(arg)
        func_method_pattern = r'(FS|NFS|T)\s*\(\s*([xm]\d+)\s*\)->(\w+)\s*\(([^)]*)\)'
        for match in re.finditer(func_method_pattern, lhs):
            base, var, method, arg = match.groups()
            
            # Elemento sin dirección (normalizado)
            element = f"{base}()->{method}()"
            elements.append(element)
            
            # Con dirección: incluye qué variable es receptor y cuál es argumento
            if arg:
                # Extraer variable del argumento si existe
                arg_var_match = re.search(r'([xwm]\d+)', arg)
                if arg_var_match:
                    arg_var = arg_var_match.group(1)
                    direction = f"{var}->{arg_var}"
                else:
                    direction = f"{var}->literal"
            else:
                direction = var
            
            signed.append((element, direction))
        
        # Operaciones tipo FS(x)->method() sin argumento complejo
        func_simple_pattern = r'(FS|NFS|T)\s*\(\s*([xm]\d+)\s*\)->(\w+)\s*\(\)'
        for match in re.finditer(func_simple_pattern, lhs):
            base, var, method = match.groups()
            # Evitar duplicados con el patrón anterior
            full_match = match.group(0)
            if full_match not in lhs or f"{base}({var})->{method}()" in [s[0].replace("()", f"({var})") for s in signed]:
                continue
            
            element = f"{base}()->{method}()"
            if element not in elements:
                elements.append(element)
                signed.append((element, var))
        
        # Operaciones tipo FS(x) sin método (comparación directa)
        func_only_pattern = r'(FS|NFS|T)\s*\(\s*([xm]\d+)\s*\)(?!->)'
        for match in re.finditer(func_only_pattern, lhs):
            base, var = match.groups()
            element = f"{base}()"
            if element not in elements:
                elements.append(element)
                signed.append((element, var))
        
        # Operaciones sobre variables de palabra: w1->last(), w1->size()
        word_pattern = r'(w\d+)->(\w+)\s*\(([^)]*)\)'
        for match in re.finditer(word_pattern, lhs):
            var, method, arg = match.groups()
            
            element = f"w->{method}()"
            elements.append(element)
            
            # Dirección: incluye la variable y argumento si existe
            if arg:
                arg_clean = arg.strip().strip("'\"")
                if re.match(r'[xwm]\d+', arg):
                    direction = f"{var}->{arg}"
                elif arg_clean:
                    direction = f"{var}->'{arg_clean}'"
                else:
                    direction = var
            else:
                direction = var
            
            signed.append((element, direction))
        
        # Extraer operador de comparación global del LHS
        comparison_op = None
        for op in ['==', '!=', '>=', '<=', '>', '<']:
            if op in lhs:
                comparison_op = op
                break
        
        # Si hay operador, añadirlo a las operaciones signed
        if comparison_op and signed:
            signed = [(elem, f"{dir}|{comparison_op}") for elem, dir in signed]
        
        return elements, signed
    
    def extract_rhs_predicates(self, rhs: str) -> Tuple[List[str], List[Tuple[str, str]]]:
        """
        Extrae predicados del RHS.
        
        Retorna:
            - elements: lista de predicados sin variables (para J_RHS)
            - signed: lista de (predicado, firma_completa) (para J_RHS_Signed)
        """
        elements = []
        signed = []
        
        # Patrón: [¬]R(var1, var2) o [¬]Result(var1, var2)
        pred_pattern = r'(¬|!)?\s*(R|Result)\s*\(\s*([xwm]\d+)\s*,\s*([xwm]\d+)\s*\)'
        
        for match in re.finditer(pred_pattern, rhs):
            negation, pred_name, var1, var2 = match.groups()
            is_negated = bool(negation)
            
            # Elemento sin variables (normalizado)
            neg_prefix = "¬" if is_negated else ""
            element = f"{neg_prefix}{pred_name}()"
            elements.append(element)
            
            # Signed: incluye variables para distinguir R(x1,w1) de R(x2,w1)
            signature = f"{neg_prefix}{pred_name}({var1},{var2})"
            signed.append((element, signature))
        
        return elements, signed
    
    def parse_mr(self, lhs: str, rhs: str, group: str, category: str, mr_id: str) -> MetamorphicRelation:
        """Parsea una MR del dominio de predicados"""
        mr = MetamorphicRelation(
            id=mr_id, group=group, category=category,
            domain=Domain.PREDICATE,
            lhs_raw=lhs, rhs_raw=rhs,
            raw_text=f"{lhs} => {rhs}"
        )
        
        # Extraer LHS
        mr.lhs_elements, mr.lhs_signed = self.extract_lhs_operations(lhs)
        
        # Extraer RHS
        mr.rhs_elements, mr.rhs_signed = self.extract_rhs_predicates(rhs)
        
        return mr


## Unified File Loader


In [351]:
class UnifiedFileLoader:
    """Carga MRs detectando automáticamente el dominio"""
    
    def __init__(self):
        self.numeric_parser = NumericParser()
        self.predicate_parser = PredicateParser()
    
    def load(self, filepath: str) -> Tuple[List[MetamorphicRelation], Domain]:
        """Carga MRs y retorna la lista junto con el dominio detectado"""
        
        with open(filepath, 'r', encoding='utf-8') as f:
            content = f.read()
            lines = content.split('\n')
        
        filename = Path(filepath).stem
        domain = detect_domain(content, filename)
        
        mrs = []
        for i, line in enumerate(lines, 1):
            line = line.strip()
            if not line or line.startswith('#'):
                continue
            
            try:
                mr = self._parse_line(line, domain)
                if mr:
                    mrs.append(mr)
            except Exception as e:
                print(f"  Warning line {i}: {e}")
        
        return mrs, domain
    
    def _parse_line(self, line: str, domain: Domain) -> Optional[MetamorphicRelation]:
        """Parsea una línea según el dominio"""
        line = line.replace("'=>", "=>")
        
        if '=>' not in line:
            return None
        
        parts_by_arrow = line.split('=>', 1)
        left_part = parts_by_arrow[0].strip()
        rhs = parts_by_arrow[1].strip()
        
        comma_parts = [p.strip() for p in left_part.split(',')]
        
        # Detectar formato de columnas
        if len(comma_parts) >= 4:
            if re.match(r'^MR\d+$', comma_parts[1]):
                group, mr_id, category = comma_parts[0], comma_parts[1], comma_parts[2]
                lhs = ','.join(comma_parts[3:])
            else:
                group, category, mr_id = comma_parts[0], comma_parts[1], comma_parts[2]
                lhs = ','.join(comma_parts[3:])
        elif len(comma_parts) == 3:
            group = comma_parts[0]
            if re.match(r'^MR\d+$', comma_parts[1]):
                mr_id, category = comma_parts[1], 'normal'
            else:
                category, mr_id = comma_parts[1], 'MR'
            lhs = comma_parts[2]
        else:
            group, category, mr_id = 'default', 'normal', 'MR'
            lhs = left_part
        
        # Parsear según dominio
        if domain == Domain.NUMERIC:
            return self.numeric_parser.parse_mr(lhs, rhs, group, category, mr_id)
        else:
            return self.predicate_parser.parse_mr(lhs, rhs, group, category, mr_id)

# ============================================================
# SIGNATURE EXTRACTION (Unificada)
# ============================================================

def extract_signature(mr: MetamorphicRelation) -> UnifiedSignature:
    """Extrae signature unificada para cualquier dominio"""
    sig = UnifiedSignature(mr_id=mr.full_id)
    
    sig.lhs = set(mr.lhs_elements)
    sig.rhs = set(mr.rhs_elements)
    sig.lhs_signed = set(mr.lhs_signed)
    sig.rhs_signed = set(mr.rhs_signed)
    
    sig.template = f"LHS[{len(mr.lhs_elements)}]=>RHS[{len(mr.rhs_elements)}]"
    
    return sig

# ============================================================
# SIMILARITY COMPUTATION (Fórmula Unificada)
# ============================================================

def jaccard(set_a: Set, set_b: Set) -> float:
    """Índice Jaccard estándar"""
    if not set_a and not set_b:
        return 1.0
    union = len(set_a | set_b)
    return len(set_a & set_b) / union if union > 0 else 0.0

def compute_similarity(sig_a: UnifiedSignature, sig_b: UnifiedSignature) -> Dict[str, float]:
    """Calcula similaridad usando la fórmula unificada"""
    return {
        'J_LHS': jaccard(sig_a.lhs, sig_b.lhs),
        'J_RHS': jaccard(sig_a.rhs, sig_b.rhs),
        'J_LHS_Signed': jaccard(sig_a.lhs_signed, sig_b.lhs_signed),
        'J_RHS_Signed': jaccard(sig_a.rhs_signed, sig_b.rhs_signed),
        'Sim_Template': 1.0 if sig_a.template == sig_b.template else 0.0
    }

def compute_composite(views: Dict[str, float]) -> float:
    """Calcula similaridad compuesta usando pesos unificados"""
    return sum(views.get(k, 0) * WEIGHTS.get(k, 0) for k in WEIGHTS)

def compute_similarity_matrix(mrs: List[MetamorphicRelation]) -> Tuple[Dict[str, pd.DataFrame], Dict[str, UnifiedSignature]]:
    """Calcula matrices de similaridad"""
    
    mr_ids = [mr.full_id for mr in mrs]
    n = len(mrs)
    
    # Extraer signatures
    signatures = {mr.full_id: extract_signature(mr) for mr in mrs}
    
    view_names = list(WEIGHTS.keys()) + ['Composite']
    matrices = {v: np.zeros((n, n)) for v in view_names}
    
    for i, id_a in enumerate(mr_ids):
        for j, id_b in enumerate(mr_ids):
            if i == j:
                for v in view_names:
                    matrices[v][i, j] = 1.0
            else:
                views = compute_similarity(signatures[id_a], signatures[id_b])
                for v in list(WEIGHTS.keys()):
                    matrices[v][i, j] = views[v]
                matrices['Composite'][i, j] = compute_composite(views)
    
    result_dfs = {v: pd.DataFrame(matrices[v], index=mr_ids, columns=mr_ids) for v in view_names}
    
    return result_dfs, signatures


## Feature Coverage Analysis


In [353]:
def analyze_coverage(mrs: List[MetamorphicRelation], domain: Domain) -> pd.DataFrame:
    """Analiza cobertura de features"""
    
    lhs_counter = Counter()
    rhs_counter = Counter()
    
    for mr in mrs:
        for elem in mr.lhs_elements:
            lhs_counter[elem] += 1
        for elem in mr.rhs_elements:
            rhs_counter[elem] += 1
    
    all_features = set(lhs_counter.keys()) | set(rhs_counter.keys())
    
    data = []
    for feat in sorted(all_features):
        lhs_count = lhs_counter.get(feat, 0)
        rhs_count = rhs_counter.get(feat, 0)
        
        if lhs_count > 0 and rhs_count == 0:
            feat_type = 'LHS Only'
        elif rhs_count > 0 and lhs_count == 0:
            feat_type = 'RHS Only'
        else:
            feat_type = 'Both'
        
        data.append({
            'Feature': feat,
            'LHS Count': lhs_count,
            'RHS Count': rhs_count,
            'Type': feat_type
        })
    
    return pd.DataFrame(data)


## Matrix Analysis


In [355]:
def analyze_similarity_matrix(sim_matrix: pd.DataFrame, threshold: float = 0.99,
                              name: str = "Composite") -> Dict:
    """Analiza matriz de similaridad"""
    n = len(sim_matrix)
    upper_tri = np.triu_indices(n, k=1)
    values = sim_matrix.values[upper_tri]
    values_rounded = np.round(values, 2)
    
    lines = []
    lines.append("=" * 50)
    lines.append(f"MATRIX ANALYSIS: {name}")
    lines.append("=" * 50)
    lines.append(f"\nTotal pairs: {len(values)}")
    
    if len(values) > 0:
        lines.append(f"Mean: {np.mean(values):.3f} | Median: {np.median(values):.3f}")
        lines.append(f"Min: {np.min(values):.3f} | Max: {np.max(values):.3f}")
        lines.append(f"Std: {np.std(values):.3f}")
        
        lines.append("\n--- Distribution by range ---")
        ranges = [(0, 0.1), (0.1, 0.3), (0.3, 0.5), (0.5, 0.7), (0.7, 0.9), (0.9, 1.01)]
        for low, high in ranges:
            count = np.sum((values >= low) & (values < high))
            pct = 100 * count / len(values)
            label = f"[{low:.1f}-{high:.1f}):" if high <= 1.0 else "[0.9-1.0]:"
            lines.append(f"  {label:12} {count:3d} ({pct:5.1f}%)")
        
        lines.append("\n--- Top 5 most frequent values ---")
        for val, count in Counter(values_rounded).most_common(5):
            pct = 100 * count / len(values)
            lines.append(f"  {val:.2f}: {count} occurrences ({pct:.1f}%)")
    
    redundant = [(sim_matrix.index[i], sim_matrix.columns[j], sim_matrix.iloc[i,j])
                 for i in range(n) for j in range(i+1, n)
                 if sim_matrix.iloc[i,j] >= threshold]
    
    lines.append(f"\n--- Pairs >= {threshold}: {len(redundant)} ---")
    for a, b, sim in redundant:
        lines.append(f"  {a} <-> {b}: {sim:.3f}")
    
    identical = [(a, b, s) for a, b, s in redundant if s >= 0.99]
    if identical:
        lines.append(f"\n--- Near-identical pairs (>= 0.99): {len(identical)} ---")
        for a, b, s in identical:
            lines.append(f"  {a} <-> {b}: {s:.3f}")
    else:
        lines.append(f"\n--- No Near-identical pairs (>= 0.99) found: {len(identical)} ---")
        
    for line in lines:
        print(line)
    
    return {
        'n_pairs': len(values),
        'mean': np.mean(values) if len(values) > 0 else 0,
        'n_redundant': len(redundant),
        'n_identical': len(identical),
        'identical_pairs': identical,
        'redundant_pairs': redundant,
        'report_lines': lines
    }


## Pdf Generation


In [357]:
def generate_heatmap_pdf(matrix_df: pd.DataFrame, output_path: str, title: str = ""):
    """Genera heatmap PDF"""
    fig, ax = plt.subplots(figsize=HEATMAP_FIGSIZE)

    sns.heatmap(
        matrix_df,
        annot=HEATMAP_ANNOT,
        fmt=HEATMAP_FMT,
        cmap="RdYlGn",
        vmin=0,
        vmax=1,
        center=0.5,
        square=True,
        linewidths=0.5,
        ax=ax,
        cbar_kws={"shrink": 0.8},
        annot_kws={"size": 18} 
    )

    # --- Titles + ticks (primero) ---
    ax.set_title("", fontsize=14, fontweight="bold", pad=20)
    ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha="right", fontsize=8)
    ax.set_yticklabels(ax.get_yticklabels(), rotation=0, fontsize=8)

    # --- Helpers ---
    def _human_indices(labels):
        idxs = []
        for i, lbl in enumerate(labels):
            if isinstance(lbl, str) and ("human" in lbl.lower()):
                idxs.append(i)
        return idxs

    # --- Label lists (desde los Text ya puestos en el eje) ---
    xt = ax.get_xticklabels()
    yt = ax.get_yticklabels()
    xlabels = [t.get_text() for t in xt]
    ylabels = [t.get_text() for t in yt]

    x_h = _human_indices(xlabels)
    y_h = _human_indices(ylabels)

    # --- Línea de separación tras el último "human" ---
    # (vertical para columnas, horizontal para filas)
    if x_h:
        sep_x = max(x_h) + 1
        if 0 < sep_x < len(xlabels):
            ax.axvline(sep_x, color="black", linewidth=3.5, zorder=10)
    if y_h:
        sep_y = max(y_h) + 1
        if 0 < sep_y < len(ylabels):
            ax.axhline(sep_y, color="black", linewidth=3.5, zorder=10)

    # --- Estilo de ticks: "human" en negrita, 2 primeros negros, resto azules ---
    x_first2 = set(sorted(x_h)[:2])
    y_first2 = set(sorted(y_h)[:2])

    for i, t in enumerate(xt):
        txt = t.get_text()
        if isinstance(txt, str) and ("human" in txt.lower()):
            t.set_fontweight("bold")
            t.set_color("black" if i in x_first2 else "blue")
        else:
            t.set_color("black")

    for i, t in enumerate(yt):
        txt = t.get_text()
        if isinstance(txt, str) and ("human" in txt.lower()):
            t.set_fontweight("bold")
            t.set_color("black" if i in y_first2 else "blue")
        else:
            t.set_color("black")

    plt.tight_layout()
    plt.savefig(output_path, format="pdf", dpi=300, bbox_inches="tight")
    plt.close()
    print(f"  Saved: {output_path}")


def generate_coverage_pdf(coverage_df: pd.DataFrame, output_path: str, title: str = ""):
    """Genera gráfico de cobertura"""
    if len(coverage_df) == 0:
        return

    fig, axes = plt.subplots(1, 2, figsize=(14, max(6, len(coverage_df) * 0.3)))

    # LHS features
    lhs_df = coverage_df[coverage_df["LHS Count"] > 0].sort_values("LHS Count", ascending=True)
    if len(lhs_df) > 0:
        axes[0].barh(lhs_df["Feature"], lhs_df["LHS Count"], color="steelblue", alpha=0.8)
        axes[0].set_xlabel("MR Count")
        axes[0].set_title("LHS Features", fontweight="bold")

    # RHS features
    rhs_df = coverage_df[coverage_df["RHS Count"] > 0].sort_values("RHS Count", ascending=True)
    if len(rhs_df) > 0:
        axes[1].barh(rhs_df["Feature"], rhs_df["RHS Count"], color="coral", alpha=0.8)
        axes[1].set_xlabel("MR Count")
        axes[1].set_title("RHS Features", fontweight="bold")

    fig.suptitle(f"Feature Coverage - {title}", fontsize=14, fontweight="bold", y=1.02)
    plt.tight_layout()
    plt.savefig(output_path, format="pdf", dpi=300, bbox_inches="tight")
    plt.close()
    print(f"  Saved: {output_path}")


## Report Generation


In [359]:
def generate_report(filename: str, mrs: List[MetamorphicRelation], domain: Domain,
                    coverage_df: pd.DataFrame, analysis: Dict,
                    signatures: Dict[str, UnifiedSignature], output_path: str) -> List[str]:
    """Genera reporte de análisis"""
    
    lines = []
    lines.append("=" * 60)
    lines.append(f"MR ANALYSIS REPORT: {filename}")
    lines.append("=" * 60)
    lines.append(f"\nDomain: {domain.value.upper()}")
    lines.append(f"MRs loaded: {len(mrs)}")
    
    # Fórmula usada
    lines.append("\n" + "-" * 40)
    lines.append("SIMILARITY FORMULA (Unified)")
    lines.append("-" * 40)
    for metric, weight in WEIGHTS.items():
        lines.append(f"  {metric}: {weight:.3f} ({weight*100:.1f}%)")
    
    # Feature coverage
    lines.append("\n" + "-" * 40)
    lines.append("FEATURE COVERAGE")
    lines.append("-" * 40)
    if len(coverage_df) > 0:
        lines.append(coverage_df.to_string(index=False))
    else:
        lines.append("  No features extracted")
    
    # MR details
    lines.append("\n" + "-" * 40)
    lines.append("MR DETAILS")
    lines.append("-" * 40)
    for mr in mrs:
        lines.append(f"\n  {mr.full_id}:")
        lines.append(f"    LHS: {mr.lhs_raw}")
        lines.append(f"    RHS: {mr.rhs_raw}")
        sig = signatures[mr.full_id]
        lines.append(f"    LHS elements: {sig.lhs}")
        lines.append(f"    RHS elements: {sig.rhs}")
        lines.append(f"    LHS signed: {sig.lhs_signed}")
        lines.append(f"    RHS signed: {sig.rhs_signed}")
    
    # Analysis
    lines.append("\n")
    lines.extend(analysis['report_lines'])

    # Non-identical
    identical_ids = set()
    for a, b, sim in analysis['identical_pairs']:
        identical_ids.add(b)
    
    non_identical = [mr.full_id for mr in mrs if mr.full_id not in identical_ids]
    
    lines.append("\n" + "-" * 40)
    lines.append(f"NON-IDENTICAL MRs (threshold<={0.99}): {len(non_identical)}")
    lines.append("-" * 40)
    for mr_id in non_identical:
        lines.append(f"  {mr_id}")

    lines.append(f"\nIDENTICAL MRs (threshold={1.0}): {len(identical_ids)}")
    for mr_id in identical_ids:
        lines.append(f"  {mr_id}")

    lines.append("\n" + "=" * 60)

    # Non-redundant
    redundant_ids = set()
    for a, b, sim in analysis['redundant_pairs']:
        redundant_ids.add(b)
    
    non_redundant = [mr.full_id for mr in mrs if mr.full_id not in redundant_ids]
    
    lines.append("\n" + "-" * 40)
    lines.append(f"NON-REDUNDANT MRs (threshold={REDUNDANCY_THRESHOLD}): {len(non_redundant)}")
    lines.append("-" * 40)
    for mr_id in non_redundant:
        lines.append(f"  {mr_id}")
    
    lines.append("\n" + "=" * 60)
    lines.append("END OF REPORT")
    lines.append("=" * 60)
    
    with open(output_path, 'w', encoding='utf-8') as f:
        f.write('\n'.join(lines))
    
    print(f"  Saved: {output_path}")
    
    return non_redundant


## Main Processing


In [361]:
def process_file(filepath: str, output_dir: str) -> Dict:
    """Procesa un archivo de MRs"""
    filename = Path(filepath).stem
    print(f"\n{'='*60}")
    print(f"Processing: {filepath}")
    print(f"{'='*60}")
    
    loader = UnifiedFileLoader()
    mrs, domain = loader.load(filepath)
    
    print(f"Domain detected: {domain.value.upper()}")
    print(f"MRs loaded: {len(mrs)}")
    
    if len(mrs) == 0:
        return {'filename': filename, 'status': 'empty'}
    
    # Mostrar ejemplos parseados
    print("\nParsed MRs (first 3):")
    for mr in mrs[:3]:
        print(f"  {mr.full_id}:")
        print(f"    LHS elements: {mr.lhs_elements}")
        print(f"    LHS signed: {mr.lhs_signed}")
        print(f"    RHS elements: {mr.rhs_elements}")
        print(f"    RHS signed: {mr.rhs_signed}")
    
    # Calcular similaridad
    sim_dfs, signatures = compute_similarity_matrix(mrs)
    
    # Cobertura
    coverage_df = analyze_coverage(mrs, domain)
    
    # Generar PDFs
    heatmap_path = os.path.join(output_dir, f"{filename}_similarity.pdf")
    generate_heatmap_pdf(sim_dfs['Composite'], heatmap_path,
                         title=f'Similarity')
    
    coverage_path = os.path.join(output_dir, f"{filename}_coverage.pdf")
    generate_coverage_pdf(coverage_df, coverage_path, title=filename)
    
    # Análisis
    print(f"\n--- Similarity Analysis ---")
    analysis = analyze_similarity_matrix(sim_dfs['Composite'],
                                         threshold=REDUNDANCY_THRESHOLD,
                                         name="Composite")


    # Reporte
    report_path = os.path.join(output_dir, f"{filename}_report.txt")
    non_redundant = generate_report(filename, mrs, domain, coverage_df,
                                    analysis, signatures, report_path)
    
    print(f"\nResult: {len(mrs)} MRs -> {len(non_redundant)} non-redundant")
    
    return {
        'filename': filename,
        'domain': domain.value,
        'n_mrs': len(mrs),
        'n_non_redundant': len(non_redundant),
        'status': 'success'
    }


## Execution


In [363]:
print("=" * 60)
print("MR SIMILARITY ANALYSIS - Unified Formula")
print("=" * 60)
print("\nFormula:")
for metric, weight in WEIGHTS.items():
    print(f"  {weight:.3f} * {metric}")

results = []
for fp in INPUT_FILES:
    if os.path.exists(fp):
        results.append(process_file(fp, OUTPUT_DIR))
    else:
        print(f"\nFile not found: {fp}")

print("\n" + "=" * 60)
print("SUMMARY")
print("=" * 60)
for r in results:
    if r.get('status') == 'success':
        print(f"{r['filename']} ({r['domain']}): {r['n_mrs']} MRs -> {r['n_non_redundant']} non-redundant")


MR SIMILARITY ANALYSIS - Unified Formula

Formula:
  0.100 * J_LHS
  0.100 * J_RHS
  0.375 * J_LHS_Signed
  0.375 * J_RHS_Signed
  0.050 * Sim_Template

Processing: mrs_AVs.txt
Domain detected: NUMERIC
MRs loaded: 14

Parsed MRs (first 3):
  example-human-MR1:
    LHS elements: ['NS']
    LHS signed: [('NS', '-')]
    RHS elements: ['TTD', 'D']
    RHS signed: [('TTD', '+'), ('D', '+')]
  example-human-MR2:
    LHS elements: ['OC']
    LHS signed: [('OC', '-')]
    RHS elements: ['TTD', 'D']
    RHS signed: [('TTD', '-'), ('D', '-')]
  example-detailed-MR3:
    LHS elements: ['WPC']
    LHS signed: [('WPC', '-')]
    RHS elements: ['TTD', 'D']
    RHS signed: [('TTD', '-'), ('D', '-')]
  Saved: output_similarity/mrs_AVs_similarity.pdf
  Saved: output_similarity/mrs_AVs_coverage.pdf

--- Similarity Analysis ---
MATRIX ANALYSIS: Composite

Total pairs: 91
Mean: 0.197 | Median: 0.150
Min: 0.000 | Max: 0.750
Std: 0.199

--- Distribution by range ---
  [0.0-0.1):    42 ( 46.2%)
  [0.1-0.3):